In [ ]:
# ---- Let this notebook import our code from src/ ----
import sys
from pathlib import Path

# Notebooks run from the notebooks/ folder, and the project root is one level up.
# sys.path is Python's list of places to look for imports. Adding the root makes "from src..." work
sys.path.insert(0, str(Path.cwd().parent))

# Pick up edits to src/ files automatically, without restarting the kernel
%load_ext autoreload
%autoreload 2

from src.config import load_config, ROOT
from src.rules import apply_rules

CONFIG = load_config()
print("Project root:", ROOT)                       # Should end in Projects-COD/inbox-agent
print("Types loaded:", len(CONFIG["types"]))       # Should be 14

In [ ]:
#SETTINGS AND FILE PATHS

from pathlib import Path

# The project root, where credentials.json lives. Notebooks run from /notebooks, so we point up
ROOT = Path.home() / "Projects-COD" / "inbox-agent"
CREDENTIALS_FILE = ROOT / "credentials.json"   # The app's ID card (from Google Cloud)
TOKEN_FILE = ROOT / "token.json"               # Your saved login, created on first sign-in

# The permission we ask for: read-only Gmail. Least privilege
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

print("ID card found:", CREDENTIALS_FILE.exists())   # Should be True

In [ ]:
#SIGN IN-CREDENTIALS USED TO SIGN IN
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

def get_credentials():
    """Return valid Google credentials, signing in through the browser only if needed."""
    creds = None

    # If you've signed in before, reuse the saved login
    if TOKEN_FILE.exists():
        creds = Credentials.from_authorized_user_file(str(TOKEN_FILE), SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # The short-term pass expired: quietly get a new one, no browser needed
            creds.refresh(Request())
        else:
            # First time: open the browser so you can sign in and approve
            flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES)
            creds = flow.run_local_server(port=0)   # port=0 means pick any free port on your laptop
        # Save the login so next time there's no browser step
        TOKEN_FILE.write_text(creds.to_json())

    return creds

creds = get_credentials()
print("Signed in:", creds.valid)

In [ ]:
#CREATE SERVICE AND PRINTS UNREAD EMAILS
from googleapiclient.discovery import build

# Create a connection to Gmail using your login
service = build("gmail", "v1", credentials=creds)

# .execute() actually sends the request. Before that, it's just prepared, like writing a query before running it
profile = service.users().getProfile(userId="me").execute()
inbox = service.users().labels().get(userId="me", id="INBOX").execute()

print("Signed in as:", profile["emailAddress"])
print("Unread emails in inbox:", inbox["messagesUnread"])
print("Unread conversations in inbox:", inbox["threadsUnread"])

In [ ]:
# ---- The label the agent must NEVER access ----
# Set this to the exact name shown in Gmail's left pane (capitals and spaces matter)
IMPORTANT_LABEL = "[important-documents]"

# Test the search syntax: list IDs only (no content is read) for emails carrying that label.
# If this prints 0 but Gmail shows emails under that label, the name or syntax is wrong,
# and the exclusion below would silently fail.
check = service.users().messages().list(userId="me", q=f'label:{IMPORTANT_LABEL}', maxResults=50).execute()
print("Emails found under the important label:", len(check.get("messages", [])))

In [ ]:
def survey_inbox(service, max_results=200):
    """Collect sender, subject, and the mass-mail signal for recent inbox emails (read AND unread).
    Headers only, no bodies: fast, and pulls minimal data.
    Purpose: find exact sender addresses and subject patterns for the triage rules."""

    # Search: everything in the inbox, EXCLUDING the important-documents label.
    # The minus sign means "not". Excluding at the search step means those emails are never fetched.
    query = f'in:inbox -label:"{IMPORTANT_LABEL}"'
    response = service.users().messages().list(userId="me", q=query, maxResults=max_results).execute()

    rows = []
    for msg in response.get("messages", []):
        # format="metadata" with no header list returns all headers and no body
        full = service.users().messages().get(userId="me", id=msg["id"], format="metadata").execute()

        # Lowercase header names, the same normalization as fetch_unread
        headers = {h["name"].lower(): h["value"] for h in full["payload"].get("headers", [])}

        rows.append({
            "sender": headers.get("from", ""),
            "subject": headers.get("subject", ""),
            "is_mass_mail": "list-unsubscribe" in headers,
        })
    return rows

In [ ]:
from collections import Counter   # Counter = a tally of how often each value appears

survey = survey_inbox(service, max_results=200)   # ~200 calls, so expect a minute or so

# ---- Overall picture ----
sender_counts = Counter(r["sender"] for r in survey)
mass_share = sum(r["is_mass_mail"] for r in survey) / len(survey)   # True counts as 1, False as 0
print(f"{len(survey)} emails from {len(sender_counts)} distinct senders. Mass mail: {mass_share:.0%}\n")

# ---- Top 25 senders: shows the exact addresses for our rules ----
for sender, n in sender_counts.most_common(25):
    print(f"{n:3}  {sender[:70]}")

# ---- Canvas subject lines: shows the patterns rules can match ----
print("\n--- Canvas subjects ---")
canvas = [r["subject"] for r in survey if "instructure" in r["sender"].lower() or "canvas" in r["sender"].lower()]
for s in canvas[:20]:
    print(s)

In [ ]:
# ---- Show every email flagged as mass mail, grouped by sender ----

# Keep only the rows where the flag is True.
# (This one-line loop is called a "list comprehension": "r for each r in survey, if r is mass mail")
mass = [r for r in survey if r["is_mass_mail"]]
print(f"{len(mass)} mass-mail emails out of {len(survey)}\n")

# sorted(..., key=...) orders the rows by the sender field, so emails from the same sender sit together.
# "lambda r: r['sender']" is a tiny inline function meaning "for each row, use its sender to sort"
for r in sorted(mass, key=lambda r: r["sender"]):
    print(f"{r['sender'][:40]:40} | {r['subject'][:60]}")

In [ ]:
# ---- Show every email flagged as mass mail, grouped by sender ----

# Keep only the rows where the flag is True.
# (This one-line loop is called a "list comprehension": "r for each r in survey, if r is mass mail")
mass = [r for r in survey if not r["is_mass_mail"]]
print(f"{len(mass)} mass-mail emails out of {len(survey)}\n")

# sorted(..., key=...) orders the rows by the sender field, so emails from the same sender sit together.
# "lambda r: r['sender']" is a tiny inline function meaning "for each row, use its sender to sort"
for r in sorted(mass, key=lambda r: r["sender"]):
    print(f"{r['sender'][:40]:40} | {r['subject'][:60]}")

In [ ]:
# ---- Find the exact sender lines for the rule list ----

# Words that should appear in each rule sender's name or address (all lowercase,
# because we lowercase the sender before comparing)
KEYWORDS = ["vox", "graduate student council", "gsc", "international student",
            "east wheelock", "instructure", "canvas"]

# For each survey row, keep the sender if ANY keyword appears in it.
# The {...} braces make a set, which keeps each sender only once; sorted() puts them in A-Z order
matches = sorted({r["sender"] for r in survey if any(k in r["sender"].lower() for k in KEYWORDS)})

for s in matches:
    print(s)

In [ ]:
'''
#CHECKING THE YAML FILE LOADS CORRECTLY OR NOT, IT DOES BTW
import yaml

# Read the settings file. safe_load reads plain data only and never runs code hidden in the file
CONFIG = yaml.safe_load((ROOT / "config" / "triage.yaml").read_text())

# Quick checks that the structure came through as expected
print("Types defined:", len(CONFIG["types"]))                                  # Should be 14
print("Newsletter senders:", CONFIG["rules"]["event_newsletter_senders"])
print("Canvas sender:", CONFIG["rules"]["canvas_sender"])
print("Example type:", CONFIG["types"]["event_newsletter"])
'''

In [ ]:
'''
#TO READ EMAIL FROM THE THING IN ACTUAL HEADER LIKE EXAMPLE TESTED IN LAST LINE HERE
from email.utils import parseaddr   # Python's built-in tool for reading email address lines

def _address(sender):
    """Return just the email address from a From line, in lowercase.
    Example: 'Vox Daily <Vox@Dartmouth.edu>' -> 'vox@dartmouth.edu'
    Lowercase because addresses aren't case-sensitive, but senders write them inconsistently."""
    # parseaddr splits a From line into (display name, address). [1] takes the second part, the address
    return parseaddr(sender)[1].lower()

# Quick test: should print vox@dartmouth.edu
print(_address("Vox Daily <Vox@Dartmouth.edu>"))
'''


In [ ]:
'''
def apply_rules(record, config):
    """Return the type decided by a rule, or None if no rule matches (then LLM call 1 decides).

    record = one email with at least "sender" and "subject" (from the survey or fetch_unread)
    config = the loaded triage.yaml settings (CONFIG)
    """
    rules = config["rules"]
    sender = _address(record["sender"])
    subject = record["subject"].lower()   # Lowercase so phrase matching ignores capitals

    # ---- Rule 1: known event-newsletter senders ----
    newsletter_senders = [s.lower() for s in rules["event_newsletter_senders"]]
    if sender in newsletter_senders:
        return "event_newsletter"

    # ---- Rule 2: Canvas emails, sorted by subject phrase ----
    if sender == rules["canvas_sender"].lower():
        # Check each Canvas type in the order written in the config. The first phrase that matches wins
        for canvas_type, phrases in rules["canvas_subject_phrases"].items():
            if any(p.lower() in subject for p in phrases):   # "Does any phrase appear in the subject?"
                return canvas_type
        return None   # A Canvas email no phrase matched: let LLM call 1 decide (the safe default)

    # ---- No rule matched ----
    return None
'''

In [ ]:
from collections import Counter

# Run the rules on every survey email. Each result is a pair: (type or None, the email)
results = [(apply_rules(r, CONFIG), r) for r in survey]

# ---- Tally: how many emails each rule caught ----
# "for t, _ in results" unpacks each pair; "_" means "ignore this part" (the email itself)
tally = Counter(t for t, _ in results)
for t, n in tally.most_common():
    print(f"{n:4}  {t or 'no rule (goes to LLM call 1)'}")

# ---- Up to 3 example subjects per rule type, so you can check the rules are right ----
for t in tally:
    if t:   # Skip the None group
        print(f"\n--- {t} ---")
        for s in [r["subject"] for typ, r in results if typ == t][:3]:
            print("  ", s[:80])

# ---- Canvas emails that no phrase matched: candidates for new phrases ----
canvas_addr = CONFIG["rules"]["canvas_sender"].lower()
unplaced = [r["subject"] for typ, r in results if typ is None and _address(r["sender"]) == canvas_addr]
print(f"\nCanvas emails no phrase matched: {len(unplaced)}")
for s in unplaced[:10]:
    print("  ", s[:80])